# Compiling for processing-using-DRAM with Cinnamon

This tutorial takes ordinary `linalg` kernels and compiles them into programs that
compute **inside DRAM**, then runs them on a cycle-level simulator of the processor
and the memory together.

**Processing-using-DRAM (PuD).** A DRAM subarray can compute by activating several
of its rows at once. Activating three rows together leaves the bitwise *majority*
of the three in all of them (SIMDRAM, Ambit). Majority with a row of zeros is AND
and with a row of ones is OR, and a *dual-contact cell* reads back its own
complement, which gives NOT. With those, any bitwise function can be built from
just two DRAM commands:

| command | what it does |
|---|---|
| **AAP** — activate, activate, precharge | copies one row (or the majority of a triple) into another row or row group |
| **AP** — activate, precharge | activates a triple: all three rows are left holding their majority |

A command acts on a whole row at once, so data is laid out **bit-sliced**: a row
holds one bit of many values, and an *n*-bit addition becomes a short sequence of
commands per bit position, applied to thousands of values in parallel.

**What Cinnamon does.** It lowers a `linalg` kernel to bit-sliced operations,
decides with an ILP how to spread the work over DRAM banks, expands each operation
into AAP/AP commands, and compiles the result into an x86 program in which every
DRAM command is one instruction. **gem5** (MIMDRAM's build) then runs that program:
the processor computes the row addresses and issues the commands, and the memory
model executes them with DRAM timing.

## The notebooks

| notebook | what you do |
|---|---|
| **00** (this one) | set up the Gurobi licence the mapping step needs |
| **01** | compile `vadd` stage by stage, and read every intermediate |
| **02** | run `vadd` on gem5 and read the result |
| **03** | sweep the number of banks and watch the mapping and the run time change |
| **04** | compare the compiled program with the paper's trace-driven simulation |
| **05** | a reduction (`gemv`): how partial results from many banks are combined |

Run them in order: each takes a few minutes at most. Everything they write goes to
`tutorial/artifacts/`, named after the kernel, the memory and the bank count, so
you can open any intermediate file from the file browser.

## What is in this image

| tool | used for |
|---|---|
| `cinm-opt` | Cinnamon's compiler driver: every MLIR step |
| `mlir-translate`, `opt`, `llc` | LLVM 19: from the LLVM dialect to an object file |
| `pud-gcc` | links a kernel with its runtime (gcc 4.8, see notebook 02) |
| `gem5-pud` | MIMDRAM's gem5, with DRAM row commands in the ISA |
| `rowop-sanity` | prints the command sequence behind every primitive |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # tutorial/, for _infra
from _infra import pud
import shutil
for tool in ("cinm-opt", "llc", "pud-gcc", "gem5-pud"):
    print(f"{tool:10} ->", shutil.which(tool) or "not on PATH (see tutorial/_infra/pud.py)")

## The Gurobi licence

The mapping step (notebook 01, step 2) solves an integer linear program with
**Gurobi**. The image contains Gurobi but not a licence, and it needs one the
C++ API accepts. **An academic WLS licence** is the one that works in a container
and on Binder, and it is free for academic use:

1. Create an account at [gurobi.com](https://www.gurobi.com) with your academic e-mail.
2. In the [licence portal](https://portal.gurobi.com), request an **Academic WLS License**.
3. Download its `gurobi.lic`.
4. Upload it to your **home directory** here: the file browser on the left,
   top level, then the upload button.

Two licences that **do not** work here: a *named-user* academic licence (it is
locked to the machine you ran `grbgetkey` on) and the limited licence that comes
with `pip install gurobipy` (only the Python interface may use it).

In [ ]:
lic = pud.licence_file()
print("licence file:", lic if lic else "none found -- upload gurobi.lic to your home directory")

The file being there is not the same as it working, so let us run the mapping step
once on a small case. A WLS licence is checked online, which takes a few seconds
the first time.

In [ ]:
bits = pud.to_bits(pud.kernel("vadd_1048576"), "licence_check")
try:
    m = pud.map_banks(bits, "licence_check", banks=8)
    print("Gurobi works. It solved a model with", m.model)
except pud.LicenceError as e:
    print(e)

If that printed *Gurobi works*, continue with **01_vadd_pipeline**.